In [1]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

2026-01-26 08:22:27.904710: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-26 08:22:28.088531: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-26 08:22:28.162854: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-26 08:22:28.185995: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-26 08:22:28.304011: I tensorflow/core/platform/cpu_feature_guar

In [2]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [3]:
layers = [4096, 2048, 1]
epochs = 50
act_func = tf.nn.relu
dropout = 0.5
input_dropout = 0.2
eta = 1e-5
norm = 'tanh'

In [4]:
X_tr, X_val, _, _, y_tr, y_val, _, _ = load(norm=norm)
print("Training data shape:", X_tr.shape)
print("Validation data shape:", X_val.shape)
print("NaN in X_tr:", np.isnan(X_tr).any())
print("NaN in y_tr:", np.isnan(y_tr).any())
print("Inf in X_tr:", np.isinf(X_tr).any())
print("Inf in y_tr:", np.isinf(y_tr).any())

Training data shape: (13884, 7063)
Validation data shape: (4614, 7063)
NaN in X_tr: True
NaN in y_tr: False
Inf in X_tr: False
Inf in y_tr: False


In [5]:
model = Sequential()
for i in range(len(layers)):
    if i == 0:
        model.add(Dense(
            layers[i],
            input_shape=(X_tr.shape[1],),
            activation=act_func,
            kernel_initializer='he_normal'
        ))
        model.add(Dropout(float(input_dropout)))
    elif i == len(layers) - 1:
        model.add(Dense(
            layers[i],
            activation='linear',
            kernel_initializer="he_normal"
        ))
    else:
        model.add(Dense(
            layers[i],
            activation=act_func,
            kernel_initializer="he_normal"
        ))
        model.add(Dropout(float(dropout)))

I0000 00:00:1769415758.039083     223 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1769415758.345934     223 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1769415758.345967     223 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1769415758.348120     223 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1769415758.348163     223 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [6]:
model.compile(
    loss='mean_squared_error',
    optimizer=K.optimizers.SGD(
        learning_rate=float(eta),
        momentum=0.5
    )
)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 4096)              28934144  
                                                                 
 dropout (Dropout)           (None, 4096)              0         
                                                                 
 dense_1 (Dense)             (None, 2048)              8390656   
                                                                 
 dropout_1 (Dropout)         (None, 2048)              0         
                                                                 
 dense_2 (Dense)             (None, 1)                 2049      
                                                                 
Total params: 37326849 (142.39 MB)
Trainable params: 37326849 (142.39 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
hist = model.fit(
    X_tr, y_tr,
    epochs=epochs,
    batch_size=64,
    shuffle=True,
    validation_data=(X_val, y_val),
    verbose=1   
)

Epoch 1/50


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

 17/217 [=>............................] - ETA: 0s - loss: nan  

2026-01-26 08:22:41.118309: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90701
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
I0000 00:00:1769415761.148909     336 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx8

217/217 [==============================] - 3s 5ms/step - loss: nan - val_loss: nan
Epoch 2/50
217/217 [==============================] - 1s 3ms/step - loss: nan - val_loss: nan
Epoch 3/50
217/217 [==============================] - 1s 4ms/step - loss: nan - val_loss: nan
Epoch 4/50
217/217 [==============================] - -0s -851us/step - loss: nan - val_loss: nan
Epoch 5/50
217/217 [==============================] - -0s -612us/step - loss: nan - val_loss: nan
Epoch 6/50
217/217 [==============================] - 1s 3ms/step - loss: nan - val_loss: nan
Epoch 7/50
217/217 [==============================] - 1s 3ms/step - loss: nan - val_loss: nan
Epoch 8/50
217/217 [==============================] - 1s 3ms/step - loss: nan - val_loss: nan
Epoch 9/50
217/217 [==============================] - 1s 3ms/step - loss: nan - val_loss: nan
Epoch 10/50
217/217 [==============================] - 1s 3ms/step - loss: nan - val_loss: nan
Epoch 11/50
217/217 [==============================] - 1s 3ms/

In [8]:
val_loss = hist.history['val_loss']
train_loss = hist.history['loss']
print("Final training loss:", train_loss[-1])
print("Final validation loss:", val_loss[-1])
model.reset_states()

Final training loss: nan
Final validation loss: nan
